# TensiMenu Model 05 — Composite Score (Cosine × DASH Re-ranking)

**Author**: Najwa Kurnia

**Pendekatan**:
- Stage 1: Cosine similarity (kandidat besar, top-50)
- Stage 2: **Re-ranking dengan composite score** = α × similarity + (1-α) × normalized_dash_score
- α = 0.4 (60% bobot DASH Score, 40% bobot similarity)

**Hipotesis**: Cosine similarity sendiri kadang merekomendasikan makanan yang "mirip target" tapi belum tentu yang paling sehat. Dengan menggabungkan similarity dengan DASH Score sebagai bobot kesehatan, rekomendasi top akan lebih *clinically meaningful*.

Ini pendekatan **two-stage retrieval-and-rerank** yang umum di sistem rekomendasi modern.

---

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    DRIVE_BASE = '/content/drive/MyDrive/TensiMenu_ML'
    print(f'Google Drive mounted. Base: {DRIVE_BASE}')
except ImportError:
    IN_COLAB = False
    DRIVE_BASE = None
    print('Bukan di Colab — menggunakan path lokal.')

In [ ]:
import json, random, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

ARTIFACTS_DIR = Path(f'{DRIVE_BASE}/artifacts_v5') if IN_COLAB else Path('artifacts_v5')
ARTIFACTS_DIR.mkdir(exist_ok=True)
MODEL_VERSION = '5.0.0-composite-rerank'
ALPHA = 0.4  # Bobot similarity vs DASH score
print(f'Model: {MODEL_VERSION} | Alpha (similarity weight): {ALPHA}')

In [ ]:
DATA_PATH = f'{DRIVE_BASE}/datasets/TKPI_2017_dataset  .csv' if IN_COLAB else '../datasets/TKPI_2017_dataset  .csv'
df_raw = pd.read_csv(DATA_PATH)
for col in ['PROTEIN_g', 'KALSIUM_mg', 'NATRIUM_mg']:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

df = df_raw.rename(columns={
    'KODE': 'food_code', 'NAMA_BAHAN': 'name', 'KATEGORI': 'category',
    'ENERGI_kal': 'energy_kcal', 'LEMAK_g': 'fat_total_g', 'SERAT_g': 'fiber_g',
    'KALSIUM_mg': 'calcium_mg', 'NATRIUM_mg': 'sodium_mg', 'KALIUM_mg': 'potassium_mg',
})

DASH_FEATURES = ['sodium_mg', 'potassium_mg', 'calcium_mg', 'fiber_g', 'fat_total_g']
NUTRIENT_WEIGHTS = {
    'sodium_mg':    {'direction': 'lower',  'weight': 0.30},
    'potassium_mg': {'direction': 'higher', 'weight': 0.25},
    'calcium_mg':   {'direction': 'higher', 'weight': 0.20},
    'fiber_g':      {'direction': 'higher', 'weight': 0.15},
    'fat_total_g':  {'direction': 'lower',  'weight': 0.10},
}
RELEVANT = ['Serealia', 'Umbi Berpati', 'Kacang & Biji', 'Sayuran',
            'Buah', 'Daging & Unggas', 'Ikan, Kerang & Udang', 'Telur', 'Susu']

df = df[df['category'].isin(RELEVANT)].copy()
df = df[df[DASH_FEATURES].notna().sum(axis=1) >= 3].copy()
for f in DASH_FEATURES:
    df[f] = df.groupby('category')[f].transform(lambda s: s.fillna(s.median()))
    df[f] = df[f].fillna(df[f].median()).clip(lower=0)
df_clean = df.reset_index(drop=True)

X = df_clean[DASH_FEATURES].to_numpy(dtype=np.float64)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'Dataset bersih: {len(df_clean)} item')

## 1. DASH Score per Item (untuk Reference Profile)

Hitung DASH Score untuk profil referensi (laki-laki dewasa sehat) sebagai proxy "makanan secara umum sehat".
Ini menjadi base score yang dipakai di semua user — di produksi bisa dihitung per user.

In [ ]:
def calculate_personal_targets(profile):
    w, h, age, g = profile['weight_kg'], profile['height_cm'], profile['age'], profile['gender']
    bmr = (10*w) + (6.25*h) - (5*age) + (5 if g == 'laki-laki' else -161)
    t = {'sodium_mg': 2300.0, 'potassium_mg': 4000.0,
         'calcium_mg': 1200.0 if age > 50 else 1000.0,
         'fiber_g': 38.0 if g == 'laki-laki' else 25.0,
         'fat_total_g': round(bmr * 0.27 / 9, 1)}
    if 'ckd' in profile.get('comorbidities', []):
        t['sodium_mg'] = 1500.0; t['potassium_mg'] = 2000.0
    if profile.get('systolic_bp', 0) >= 150:
        t['sodium_mg'] = 1500.0
    return t

def calculate_dash_score(nutrition, targets):
    total = 0.0
    for n, c in NUTRIENT_WEIGHTS.items():
        actual = nutrition.get(n, 0.0)
        target = targets.get(n, 1.0)
        if target <= 0:
            cont = 0.0
        elif c['direction'] == 'higher':
            cont = min(actual / target, 1.0)
        else:
            cont = 1.0 if actual <= target else max(0.0, 1.0 - (actual - target) / target)
        total += cont * c['weight']
    return round(total * 100, 1)

## 2. Composite Re-ranking

In [ ]:
def recommend_composite(profile, top_k=15, alpha=ALPHA, candidate_size=50):
    """
    Two-stage retrieval:
    1. Stage 1: cosine similarity → top-N kandidat
    2. Stage 2: re-ranking dengan composite_score = α*sim + (1-α)*dash_normalized
    """
    targets = calculate_personal_targets(profile)
    user_vec = np.array([targets[f] for f in DASH_FEATURES])
    user_scaled = scaler.transform(user_vec.reshape(1, -1))
    
    # Stage 1: similarity
    sims = cosine_similarity(user_scaled, X_scaled)[0]
    
    # Stage 2: hitung DASH Score per kandidat
    df_work = df_clean.copy()
    df_work['similarity'] = sims
    
    # Top candidate untuk re-ranking
    candidates = df_work.nlargest(candidate_size, 'similarity').copy()
    
    # Hitung DASH Score per kandidat untuk profil ini
    candidates['dash_score'] = candidates.apply(
        lambda r: calculate_dash_score(
            {f: r[f] for f in DASH_FEATURES}, targets
        ), axis=1
    )
    
    # Normalisasi similarity dan dash_score ke [0,1] untuk komposit fair
    sim_norm = (candidates['similarity'] - candidates['similarity'].min()) / (candidates['similarity'].max() - candidates['similarity'].min() + 1e-9)
    dash_norm = candidates['dash_score'] / 100.0
    
    # Composite score
    candidates['composite_score'] = alpha * sim_norm + (1 - alpha) * dash_norm
    
    return candidates.nlargest(top_k, 'composite_score')[
        ['food_code', 'name', 'category', 'similarity', 'dash_score', 'composite_score']
    ].reset_index(drop=True)

profile = {'gender': 'laki-laki', 'weight_kg': 70, 'height_cm': 170, 'age': 45,
           'comorbidities': [], 'systolic_bp': 140}
recs = recommend_composite(profile, top_k=15)
print('=== TOP 15 (Composite Re-ranking) ===')
print(f'Alpha = {ALPHA} (similarity={ALPHA*100:.0f}%, DASH={(1-ALPHA)*100:.0f}%)')
recs

In [ ]:
# Bandingkan: cosine-only vs composite — apakah ranking berbeda?
print('\n=== COSINE-ONLY (Stage 1) ===')
user_vec = np.array([calculate_personal_targets(profile)[f] for f in DASH_FEATURES])
user_scaled = scaler.transform(user_vec.reshape(1, -1))
sims = cosine_similarity(user_scaled, X_scaled)[0]
df_compare = df_clean.copy()
df_compare['similarity'] = sims
print(df_compare.nlargest(10, 'similarity')[['name', 'similarity']].reset_index(drop=True))

In [ ]:
joblib.dump(scaler, ARTIFACTS_DIR / 'scaler.pkl')
np.save(ARTIFACTS_DIR / 'item_matrix.npy', X_scaled)
with open(ARTIFACTS_DIR / 'food_ids.json', 'w', encoding='utf-8') as f:
    json.dump(df_clean['food_code'].tolist(), f, ensure_ascii=False)

metadata = {
    'version': MODEL_VERSION,
    'approach': 'Two-stage Retrieval-and-Rerank (Cosine + DASH composite)',
    'trained_at': datetime.utcnow().isoformat() + 'Z',
    'random_state': RANDOM_STATE,
    'n_items': len(df_clean),
    'features': DASH_FEATURES,
    'alpha': ALPHA,
    'candidate_size': 50,
    'reranking_formula': 'composite = alpha * sim_norm + (1-alpha) * dash_norm',
}
with open(ARTIFACTS_DIR / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
df_clean.to_csv(ARTIFACTS_DIR / 'food_items_clean.csv', index=False)
print('✓ Artefak v5 tersimpan')